In [1]:
import os
from pathlib import Path

import polars as pl

In [2]:
DATABASE_URI = os.environ["DATABASE_URI"]

In [30]:
RANDOM_SEED = 42

In [3]:
DATASETS_PATH = Path(
    "/Users/luis/projets/beta.gouv/qfdmod/quefairedemesobjets/data-platform/notebooks/deduplication/datasets"
)

# Annotation manuelle (Christian)

In [4]:
def create_entity_pairs_from_datasets(datasets_folder_path: Path) -> pl.DataFrame:
    csv_files_to_load = datasets_folder_path.glob("*.csv")

    dfs = []
    for csv_filepath in csv_files_to_load:
        df = (
            pl.read_csv(csv_filepath).lazy().rename({"Good ? ": "Good ?"}, strict=False)
        )
        df_pairs = (
            df.join(df, on="cluster_id")
            .filter(pl.col("Good ?").is_in(["oui", "non"]))
            .with_columns(
                pl.min_horizontal(
                    pl.col("identifiant_unique"), pl.col("identifiant_unique_right")
                ).alias("identifiant_unique_i"),
                pl.max_horizontal(
                    pl.col("identifiant_unique"), pl.col("identifiant_unique_right")
                ).alias("identifiant_unique_j"),
                (pl.col("Good ?") == "oui").alias("label"),
            )
            .filter(
                (pl.col("identifiant_unique_i") != pl.col("identifiant_unique_j"))
                & (
                    pl.col("cluster_id")
                    .cum_count()
                    .over(
                        ["cluster_id", "identifiant_unique_i", "identifiant_unique_j"]
                    )
                    == 1
                )
            )
            .select(["identifiant_unique_i", "identifiant_unique_j", "label"])
        )
        dfs.append(df_pairs)

    df_pairs_concat = (
        pl.concat(dfs)
        .unique(["identifiant_unique_i", "identifiant_unique_j"])
        .sort(["identifiant_unique_i", "identifiant_unique_j"])
        .collect()
    )

    return df_pairs_concat

In [5]:
df_manual_labeling = create_entity_pairs_from_datasets(DATASETS_PATH)

In [6]:
df_manual_labeling

identifiant_unique_i,identifiant_unique_j,label
str,str,bool
"""07e07779-388e-5971-947b-b999ae…","""lokki_67692fc641d8badbf051ec32""",true
"""085352c0-77c6-599d-b54a-02ef2a…","""772950c8-5202-5394-b0ae-8ebbda…",false
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5487990001""",false
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488009001""",false
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488015001""",false
…,…,…
"""reseau_vrac_reemploi_14u_47.55…","""reseau_vrac_reemploi_1EK_47_-2""",true
"""reseau_vrac_reemploi_1DB_45_4""","""reseau_vrac_reemploi_tR_45.118…",true
"""reseau_vrac_reemploi_wA_44.818…","""reseau_vrac_reemploi_wC_44.900…",false


In [31]:
df_manual_labeling.group_by("label").len()

label,len
bool,u32
true,39
false,130


# Depuis la base

## Examples négatifs via changement de parent dont l'ancien parent existe toujours

**Vrais négatifs via changement de parent :** Acteurs qui ne sont pas des parents et dont le parent lié aujourd'hui diffère de celui trouvé dans les suggestions de clustering.

In [11]:
sql_neg_method_1 = """
with suggestions_avec_creation_de_parent as 
(
select
	ds.id,
	ds.statut,
	jsonb_path_query_array(ds.contexte, '$.fuzzy_details[*].identifiant_unique') as identifiant_unique_list,
	ds.suggestion->'changes'->0->'model_params'->>'id' as id_parent,
	ds.suggestion->'changes'->0->>'reason' as suggestion_reason,
	ds.modifie_le
from
	data_suggestion ds
inner join data_suggestioncohorte ds2 
on
	ds.suggestion_cohorte_id = ds2.id
	and ds2.type_action = 'CLUSTERING'
where
	ds.contexte->'fuzzy_details' is not null
	and ds.suggestion->'changes'->0->>'reason' in ('1️⃣ 1 seul parent existant → à garder','➕ Pas de parent → à créer')
	and ds.statut='SUCCES'
),
suggestions_explosees as (
select
	s.id,
	s.id_parent,
	s.suggestion_reason,
	s.modifie_le,
	s_lid.id_acteur
from
	suggestions_avec_creation_de_parent s,
	 jsonb_array_elements_text(s.identifiant_unique_list) as s_lid(id_acteur)
where id_acteur::text<>s.id_parent
),
suggestions_avec_parent_existant as (
SELECT 
	se.*
from suggestions_explosees se
inner join qfdmo_vueacteur qv on se.id_parent=qv.identifiant_unique
),
suggestions_vrais_negatifs as (
select
	 qv.identifiant_unique,
	 qv."uuid",
	 se.id_parent as suggestion_parent_id,
	 se.id as suggestion_id,
	 se.suggestion_reason
from
	qfdmo_vueacteur qv
inner join suggestions_avec_parent_existant se on qv.identifiant_unique = se.id_acteur and qv.parent_id!=se.id_parent
where
	not qv.est_parent
	and qv.statut <> 'SUPPRIME'
	and qv.modifie_le <= now() - interval '1 day'
)
select
	identifiant_unique as identifiant_unique_i,
	suggestion_parent_id as identifiant_unique_j
from suggestions_vrais_negatifs
union all
select
	svn.identifiant_unique as identifiant_unique_i,
	qv.identifiant_unique as identifiant_unique_j
from suggestions_vrais_negatifs svn
inner join 	qfdmo_vueacteur qv
on svn.suggestion_parent_id=qv.parent_id and svn.identifiant_unique!=qv.identifiant_unique
"""

In [18]:
df_negatives_via_parent_change = pl.read_database_uri(
    query=sql_neg_method_1, uri=DATABASE_URI
).with_columns(pl.lit(False).alias("label"))

In [19]:
df_negatives_via_parent_change

identifiant_unique_i,identifiant_unique_j,label
str,str,bool
"""ocab_valobat_DCT_38100_GREN_01""","""e1b4bebc-1ff1-5a3f-b2bb-1b18bf…",false
"""ocab_ecomaison_DCT_38100_GREN_…","""e1b4bebc-1ff1-5a3f-b2bb-1b18bf…",false
"""cyclevia_decb5827-1c63-4856-9c…","""e1b4bebc-1ff1-5a3f-b2bb-1b18bf…",false
"""ocab_ecominero_DCT_38100_GREN_…","""e1b4bebc-1ff1-5a3f-b2bb-1b18bf…",false
"""ocab_valdelia_DCT_38100_GREN_0…","""e1b4bebc-1ff1-5a3f-b2bb-1b18bf…",false
…,…,…
"""ocab_valdelia_DCT_38100_GREN_0…","""aliapur_21141""",false
"""ocab_valdelia_DCT_38100_GREN_0…","""ademesinoedecheteries_4130""",false
"""ocab_valdelia_DCT_38100_GREN_0…","""ocab_valobat_DCT_38021_GREN_00""",false


# Par échantillonage aléatoire

**Vrai négatifs :** le but est de créer des paires d'acteurs qui ne sont pas reliées entre elles.
Contraintes :

- Pas de parent en commun
- Pas de relation parent<->enfant
- Pas la même source
- Distance géographique > 10km OU pas le même département

In [25]:
sql_neg_method_2 = """
WITH
  sampled AS (
    SELECT
      identifiant_unique,
      parent_id,
      source_id,
      "location",
      code_postal,
      row_number() OVER (
        ORDER BY
          MD5(identifiant_unique::TEXT || '42')
      ) AS rn -- Seed pour le reproductibilité
    FROM
      qfdmo_vueacteur qv
    WHERE
      qv.statut <> 'SUPPRIME'
  ),
  sampled_filtered AS (
    SELECT
      *
    FROM
      sampled
    WHERE
      rn <= 100000 -- Permet la création d'environ 50k paires
  ),
  randomized_pairs AS (
    SELECT
      identifiant_unique,
      parent_id,
      source_id,
      "location",
      code_postal,
      row_number() OVER (
        ORDER BY
          MD5(identifiant_unique::TEXT || '42')
      ) AS rn -- Mélange aléatoire REPRODUCTIBLE de l'échantillon
    FROM
      sampled_filtered
  )
  -- Création des paires
SELECT
  MIN(a.identifiant_unique) AS identifiant_unique_i,
  MAX(b.identifiant_unique) AS identifiant_unique_j
FROM
  randomized_pairs a
  JOIN randomized_pairs b ON FLOOR((a.rn - 1) / 2) = FLOOR((b.rn - 1) / 2) -- Groupe en paire en utilisant le ~row_number/2 comme clé
  AND a.identifiant_unique < b.identifiant_unique -- S'assure de ne pas avoir de paires avec les mêmes ids dans un ordre différent
  AND a.identifiant_unique <> coalesce(b.parent_id, '') -- acteur_j n'est pas un parent de acteur_i
  AND b.identifiant_unique <> coalesce(a.parent_id, '') -- acteur_i n'est pas un parent de acteur_j
  AND coalesce(a.parent_id, 'N/A') != coalesce(b.parent_id, 'N/A2') -- les deux acteurs n'ont pas le même parent
  AND coalesce(a.source_id, -1) != coalesce(b.source_id, -2) -- les deux acteurs n'ont pas la même source
  AND (
    ST_DistanceSpheroid (a."location"::geometry, b."location"::geometry) >= 10000
    OR left(a.code_postal, 2) != left(b.code_postal, 2)
  ) -- plus de 10km entre les deux ou département différent
GROUP BY
  FLOOR((a.rn - 1) / 2)
"""

In [23]:
df_negatives_random = pl.read_database_uri(
    query=sql_neg_method_2, uri=DATABASE_URI
).with_columns(pl.lit(False).alias("label"))

In [24]:
df_negatives_random

identifiant_unique_i,identifiant_unique_j,label
str,str,bool
"""BAL_50.6011024, 5.6316395""","""ecopae_ECOPAEPDR00430""",false
"""9cf25f99-ef4f-59da-93fe-df885e…","""corepile_87-COL-0011_34""",false
"""1d95e3f0-249c-57d9-bb42-d4c35d…","""corepile_07-DIS-0108""",false
"""batribox_ENT120309GC""","""creati_vy_223078_reparation_06…",false
"""aliapur_151229000000012""","""refashion_TLC-REFASHION-PAV-34…",false
…,…,…
"""ecodds_FD0075""","""heliolite_175214_reparation_06…",false
"""4d8d5967-e4be-551b-99fd-f5323e…","""refashion_TLC-REFASHION-PAV-34…",false
"""batribox_ENT030101AL""","""ludotheques_798""",false


# Vrais positifs

In [26]:
sql_pos = """
with pool as 
(
select
	qv.identifiant_unique ,
	qv.parent_id,
	case
		when qv.est_parent then qv.identifiant_unique
		else qv.parent_id
	end as cleaned_parent_id
from
	qfdmo_vueacteur qv),
paires_dupliquees as (
select
	least(p.identifiant_unique, p2.identifiant_unique) as identifiant_unique_i,
	greatest(p.identifiant_unique, p2.identifiant_unique) as identifiant_unique_j
from
	pool p
inner join pool p2 on
	p.cleaned_parent_id = p2.cleaned_parent_id
	and p.identifiant_unique != p2.identifiant_unique
)
select
	identifiant_unique_i,
	identifiant_unique_j
from
	(
	select
		*,
		row_number() over (partition by identifiant_unique_i,
		identifiant_unique_j) as rn
	from
		paires_dupliquees
)
where
	rn = 1
"""

In [32]:
df_positives = pl.read_database_uri(query=sql_pos, uri=DATABASE_URI).with_columns(
    pl.lit(True).alias("label")
)

In [33]:
df_positives

identifiant_unique_i,identifiant_unique_j,label
str,str,bool
"""000082e5-9817-5374-b9e6-a42d06…","""citeo_5088173001""",true
"""000082e5-9817-5374-b9e6-a42d06…","""citeo_5088173002""",true
"""000245cf-f393-5e59-b2ff-a6e197…","""citeo_1764957001""",true
"""000245cf-f393-5e59-b2ff-a6e197…","""citeo_1764957002""",true
"""00032a9a-988b-5e2a-8eb0-b8b686…","""citeo_1784187001""",true
…,…,…
"""valdelia_836""","""valtri__histoires_sans_fin_581…",true
"""valdelia_EA_0000298""","""valdelia_PMCB_0000844""",true
"""valdelia_EA_0001073""","""valdelia_EA_0001077""",true


# Constitution d'un dataset équilibré

Même nombre d'examples positifs et négatifs. Pour commencer 1000 de chaque en privilégiant les annotations manuelles en en dernier lieu les paires issues d'un échantillonage aléatoire

In [34]:
df_pairs = pl.concat(
    [
        df_manual_labeling,
        df_negatives_via_parent_change,
        df_negatives_random.sample(
            n=(
                1000
                - len(df_manual_labeling.filter(pl.col("label").not_()))
                - len(df_negatives_via_parent_change)
            ), 
            seed=42
        ),
        df_positives.sample(
            n=(
                1000
                - len(df_manual_labeling.filter(pl.col("label")))
            )
        )
    ]
)

In [35]:
df_pairs

identifiant_unique_i,identifiant_unique_j,label
str,str,bool
"""07e07779-388e-5971-947b-b999ae…","""lokki_67692fc641d8badbf051ec32""",true
"""085352c0-77c6-599d-b54a-02ef2a…","""772950c8-5202-5394-b0ae-8ebbda…",false
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5487990001""",false
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488009001""",false
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488015001""",false
…,…,…
"""refashion_TLC-REFASHION-PAV-32…","""refashion_TLC-REFASHION-PAV-34…",true
"""refashion_TLC-REFASHION-PAV-33…","""refashion_TLC-REFASHION-PAV-34…",true
"""refashion_TLC-REFASHION-PAV-33…","""refashion_TLC-REFASHION-PAV-34…",true


In [36]:
df_pairs.group_by("label").len()

label,len
bool,u32
true,1000
false,1000


# Sauvegarde

In [37]:
df_pairs.write_csv(DATASETS_PATH/"dataset_20260706.csv")